## Startup notebook for energy calibration by hand

SISEPUEDE models 4 emission sectors, and 1 sector used to coordinate shared drivers among the emission models (socioeconomic):

* Agriculture, Forerstry and Land Use (AFOLU)
* Circular Economy
* Energy
    * Energy Consumption
    * Energy Production - Electricity and Fuel Production
* Industrial Processes and Product Use (IPPU)
* Socioeconomic

#### `EnergyConsumption`
We are interested on energy consumption. So we'll run the model, and extract the outputs.

First things first, `EnergyConsumption` subsectors:
* Energy Fuels (ENFU)
* Fugitive Emissions (FGTV)
* Industrial Energy (INEN)
* Transportation (TRNS)
* Transportation Demand (TRDE)
* Stationary Combustion and Other Energy (SCOE)


Actually, `EnergyConsumption` depends on on AFOLU, IPPC and CircularEconomy models. i.e. we need other models' outputs (plop).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
path = "/Users/dianamendez/sisepuede"
if path not in sys.path:
    sys.path.append(path)

import importlib
import matplotlib.pyplot as plt
import numpy as np
import os, os.path
import pandas as pd
import pathlib
import sisepuede.visualization.plots as spl
import sisepuede.utilities._plotting as sup
import sys
from sisepuede.manager.sisepuede_file_structure import SISEPUEDEFileStructure

In [2]:
from sisepuede.models.afolu import AFOLU
from sisepuede.models.circular_economy import CircularEconomy
from sisepuede.models.energy_consumption import EnergyConsumption
from sisepuede.models.ippu import IPPU

# ?EnergyConsumption

In [3]:
# initialize a file structure and get model attributes
file_struct = SISEPUEDEFileStructure()
model_attributes = file_struct.model_attributes

In [4]:
# now we can initialize the model(s) using the model_attributes object argument
model_afolu     = AFOLU             (model_attributes, )
model_circecon  = CircularEconomy   (model_attributes, )
model_ippu      = IPPU              (model_attributes, )
model_energycon = EnergyConsumption (model_attributes, )

We still haven't done anything. We'll just look at what the model would throw.

In [5]:
model_attributes.get_sector_variables(
    "Energy",
    var_type = "output",
)

out_vars_scoe = model_attributes.get_variable("Biomass Emissions from SCOE")


In [7]:
df_input = pd.read_csv("/Users/dianamendez/Documents/sisepuede/sisepuede_raw_inputs_latest_LBY_modified_march_2026.csv")
# df_input = pd.read_csv("/Users/dianamendez/sisepuede/sisepuede/ref/examples/input_data_frame.csv")
df_input.head()

,year,ef_ippu_tonne_nf3_per_tonne_production_chemicals,ef_ippu_tonne_nf3_per_tonne_production_electronics,ef_ippu_tonne_sf6_per_mmm_gdp_other_product_manufacturing,ef_ippu_tonne_sf6_per_tonne_production_chemicals,ef_ippu_tonne_sf6_per_tonne_production_electronics,ef_ippu_tonne_sf6_per_tonne_production_metals,frac_agrc_bevs_and_spices_cl2_dry,frac_agrc_cereals_cl2_dry,frac_agrc_fibers_cl2_dry,...,nemomod_entc_scalar_availability_factor_pp_nuclear,nemomod_entc_scalar_availability_factor_pp_ocean,nemomod_entc_scalar_availability_factor_pp_oil,nemomod_entc_scalar_availability_factor_pp_solar,nemomod_entc_scalar_availability_factor_pp_waste_incineration,nemomod_entc_scalar_availability_factor_pp_wind,iso_alpha_3,population_gnrl_rural,population_gnrl_urban,gdp_mmm_usd
0,2015,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,1.0,1.0,1.0,1.0,1.0,LBY,795843.0,5735976.0,116.940831
1,2016,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,1.0,1.0,1.0,1.0,1.0,LBY,808058.0,5824068.0,112.487607
2,2017,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,1.0,1.0,1.0,1.0,1.0,LBY,821006.0,5917764.0,121.473195
3,2018,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,1.0,1.0,1.0,1.0,1.0,LBY,834319.0,6014736.0,129.305468
4,2019,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,1.0,1.0,1.0,1.0,1.0,LBY,846496.0,6104537.0,112.565781


* `Circular Economy` depends on `AFOLU`
* `EnergyConsumption` depends on `AFOLU` and `IPPU` (and on `CircularEconomy`?)

In [8]:
# apply model to input data. get outputs.
df_out_afolu = model_afolu(df_input)
df_out_ippu = model_ippu(df_input)

df_input_circecon = pd.merge(df_input, df_out_afolu)
df_out_circecon = model_circecon(df_input_circecon)

df_input_energycon = pd.merge(df_input, df_out_afolu)
df_input_energycon = pd.merge(df_input_energycon, df_out_ippu)
df_out_energycon = model_energycon(df_input_energycon)


In [9]:
# list the subsectors in Energy
model_attributes.get_sector_subsectors("Energy")

['Carbon Capture and Sequestration',
 'Energy Fuels',
 'Energy Storage',
 'Energy Technology',
 'Fugitive Emissions',
 'Industrial Energy',
 'Stationary Combustion and Other Energy',
 'Transportation',
 'Transportation Demand']

In [20]:
# subsectors = ['Carbon Capture and Sequestration',
#  'Energy Fuels',
#  'Energy Storage',
#  'Energy Technology',
#  'Fugitive Emissions',
#  'Industrial Energy',
#  'Stationary Combustion and Other Energy',
#  'Transportation',
#  'Transportation Demand']

subsectors = ["Carbon Capture and Sequestration",
              "Energy Fuels", 
              "Industrial Energy",
              "Stationary Combustion and Other Energy",
              "Transportation"]

substring = "energy"
subsector_var_dir = {}

for subsector in subsectors:
    # print(subsector)
    temp_subsector_vars = model_attributes.get_subsector_variables(subsector, var_type = "output")
    subsector_vars = [var for var in temp_subsector_vars if substring in var.lower()]
    subsector_var_dir[subsector] = subsector_vars

display(subsector_var_dir)

{'Carbon Capture and Sequestration': ['Electrical Energy Consumption from CCSQ',
  'Energy Consumption from CCSQ',
  'Total Electrical Energy Consumption from CCSQ',
  'Total Energy Consumption from CCSQ'],
 'Energy Fuels': ['Energy Demand by Fuel in CCSQ',
  'Energy Demand by Fuel in Energy Technology',
  'Energy Demand by Fuel in Industrial Energy',
  'Energy Demand by Fuel in SCOE',
  'Energy Demand by Fuel in Transportation',
  'Total Energy Demand by Fuel',
  'Value of Fuel Consumed in Energy Technology',
  'Value of Fuel Consumed in Industrial Energy'],
 'Industrial Energy': [':math:\\text{CH}_4 Emissions from Industrial Energy',
  ':math:\\text{CO}_2 Biomass Emissions from Industrial Energy',
  ':math:\\text{CO}_2 Captured in Industrial Energy',
  ':math:\\text{CO}_2 Non-Biomass Emissions from Industrial Energy',
  ':math:\\text{N}_2\\text{O} Emissions from Industrial Energy',
  'Electrical Energy Consumption from Industrial Energy',
  'Energy Consumption from Industrial Energy'

In [15]:
df_out_energycon.tail()

,time_period,dem_trde_freight_mt_km,dem_trde_private_and_public_passenger_km,dem_trde_regional_passenger_km,emission_co2e_ch4_ccsq_direct_air_capture,emission_co2e_ch4_inen_agriculture_and_livestock,emission_co2e_ch4_inen_cement,emission_co2e_ch4_inen_chemicals,emission_co2e_ch4_inen_electronics,emission_co2e_ch4_inen_glass,...,vehicle_distance_traveled_trns_road_light_diesel,vehicle_distance_traveled_trns_road_light_electricity,vehicle_distance_traveled_trns_road_light_gasoline,vehicle_distance_traveled_trns_road_light_hydrocarbon_gas_liquids,vehicle_distance_traveled_trns_road_light_hydrogen,vehicle_distance_traveled_trns_water_borne,vehicle_distance_traveled_trns_water_borne_diesel,vehicle_distance_traveled_trns_water_borne_electricity,vehicle_distance_traveled_trns_water_borne_hydrogen,vehicle_distance_traveled_trns_water_borne_natural_gas
31,31,323594.579736,1.191073e+11,1.043339e+10,0.0,0.019908,0.000595,0.001617,0.000002,0.000397,...,1.646635e+10,4.746311e+08,3.091053e+10,0.0,0.0,6.155780e+06,6.155780e+06,0.0,0.0,0.0
32,32,328772.093012,1.213387e+11,1.062886e+10,0.0,0.019803,0.000592,0.001607,0.000002,0.000394,...,1.691690e+10,4.845066e+08,3.198746e+10,0.0,0.0,6.255887e+06,6.255887e+06,0.0,0.0,0.0
33,33,334032.446500,1.236120e+11,1.082799e+10,0.0,0.019697,0.000588,0.001597,0.000002,0.000392,...,1.737879e+10,4.946024e+08,3.309357e+10,0.0,0.0,6.357626e+06,6.357626e+06,0.0,0.0,0.0
34,34,339376.965644,1.259279e+11,1.103085e+10,0.0,0.019590,0.000584,0.001587,0.000002,0.000389,...,1.785228e+10,5.049231e+08,3.422958e+10,0.0,0.0,6.461024e+06,6.461024e+06,0.0,0.0,0.0
35,35,344806.997095,1.282872e+11,1.123752e+10,0.0,0.019483,0.000580,0.001576,0.000002,0.000387,...,1.833762e+10,5.154731e+08,3.539623e+10,0.0,0.0,6.566108e+06,6.566108e+06,0.0,0.0,0.0


In [16]:
df_out_energycon_filtered = df_out_energycon.loc[:, df_out_energycon.columns.str.contains("energy", case=False)]
df_out_energycon_filtered.tail()

,energy_consumption_ccsq_direct_air_capture,energy_consumption_ccsq_total,energy_consumption_electricity_ccsq_direct_air_capture,energy_consumption_electricity_ccsq_total,energy_consumption_electricity_inen_agriculture_and_livestock,energy_consumption_electricity_inen_cement,energy_consumption_electricity_inen_chemicals,energy_consumption_electricity_inen_electronics,energy_consumption_electricity_inen_glass,energy_consumption_electricity_inen_lime_and_carbonite,...,energy_demand_inen_recycled_plastic,energy_demand_inen_recycled_rubber_and_leather,energy_demand_inen_recycled_textiles,energy_demand_inen_recycled_wood,energy_demand_inen_rubber_and_leather,energy_demand_inen_textiles,energy_demand_inen_wood,energy_demand_scoe_heat_commercial_municipal,energy_demand_scoe_heat_other_se,energy_demand_scoe_heat_residential
31,0.0,0.0,0.0,0.0,4.512877,0.208868,2.188162,0.008152,0.139245,4.177363,...,0.0,0.0,0.0,0.0,3.064739,1.535452,0.059784,0.0,0.0,37.576534
32,0.0,0.0,0.0,0.0,4.489020,0.207529,2.174129,0.008100,0.138352,4.150574,...,0.0,0.0,0.0,0.0,3.057339,1.531745,0.059640,0.0,0.0,38.304442
33,0.0,0.0,0.0,0.0,4.464981,0.206198,2.160187,0.008048,0.137465,4.123958,...,0.0,0.0,0.0,0.0,3.049957,1.528047,0.059496,0.0,0.0,39.046427
34,0.0,0.0,0.0,0.0,4.440807,0.204876,2.146334,0.007996,0.136584,4.097511,...,0.0,0.0,0.0,0.0,3.042593,1.524357,0.059352,0.0,0.0,39.802808
35,0.0,0.0,0.0,0.0,4.416537,0.203562,2.132570,0.007945,0.135708,4.071235,...,0.0,0.0,0.0,0.0,3.035247,1.520677,0.059209,0.0,0.0,40.573832


In [18]:
# filter for releveant variables
# information about variables
df_variables = model_attributes.build_variable_dataframe_by_sector(
    sectors_build = ["Energy"],
    #field_subsector = "Stationary Combustion and Other Energy",
    include_model_variable = True,
    include_model_variable_attributes = True,
    include_simplex_group_as_trajgroup = False,
    include_time_periods = False,
    vartype="input"
)

df_variables_filtered = df_variables[
    df_variables["variable_field"]
    .str.contains("energy", case=False, na=False)
]

display(df_variables_filtered)

,subsector,variable_field,variable,area,energy,length,mass,monetary,power,volume,emission_gas
0,Carbon Capture and Sequestration,efficfactor_ccsq_heat_energy_direct_air_captur...,CCSQ Efficiency Factor for Heat Energy from Ge...,None,None,None,None,None,None,None,None
1,Carbon Capture and Sequestration,efficfactor_ccsq_heat_energy_direct_air_captur...,CCSQ Efficiency Factor for Heat Energy from Hy...,None,None,None,None,None,None,None,None
2,Carbon Capture and Sequestration,efficfactor_ccsq_heat_energy_direct_air_captur...,CCSQ Efficiency Factor for Heat Energy from Na...,None,None,None,None,None,None,None,None
3,Carbon Capture and Sequestration,energy_intensity_ccsq_direct_air_capture_gj_pe...,CCSQ Energy Demand Per Mass of :math:\text{CO}...,None,gj,None,tonne,None,None,None,co2
4,Carbon Capture and Sequestration,frac_ccsq_energydem_direct_air_capture_electri...,CCSQ Fraction Energy Electricity,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
965,Stationary Combustion and Other Energy,scalar_scoe_appliance_energy_demand_other_se,SCOE Appliance Energy Demand Scalar,None,None,None,None,None,None,None,None
966,Stationary Combustion and Other Energy,scalar_scoe_appliance_energy_demand_residential,SCOE Appliance Energy Demand Scalar,None,None,None,None,None,None,None,None
967,Stationary Combustion and Other Energy,scalar_scoe_heat_energy_demand_commercial_muni...,SCOE Heat Energy Demand Scalar,None,None,None,None,None,None,None,None
968,Stationary Combustion and Other Energy,scalar_scoe_heat_energy_demand_other_se,SCOE Heat Energy Demand Scalar,None,None,None,None,None,None,None,None


In [19]:
modvar = model_attributes.get_variable("SCOE Appliance Energy Demand Scalar")
display(modvar.get_from_dataframe)

modvar = model_attributes.get_variable("CCSQ Fraction Energy Electricity")
display(modvar.get_from_dataframe)

<bound method ModelVariable.get_from_dataframe of ModelVariable: SCOE Appliance Energy Demand Scalar
Fields:
	scalar_scoe_appliance_energy_demand_commercial_municipal
	scalar_scoe_appliance_energy_demand_other_se
	scalar_scoe_appliance_energy_demand_residential>

<bound method ModelVariable.get_from_dataframe of ModelVariable: CCSQ Fraction Energy Electricity
Fields:
	frac_ccsq_energydem_direct_air_capture_electricity>